# Lab 05 - Silver Layer: Data Quality, Governance & PII 
### *Clean, Enrich, Protect*

> **Note:** Fill in the `________` blanks to complete each exercise. Use the hints if you get stuck.

---

### Learning Objectives

By the end of this lab, you will be able to:

1. **Profile** data and identify quality issues before cleaning
2. **Apply** data cleaning transformations (null handling, range validation, type casting)
3. **Create** business metrics as calculated columns
4. **Enrich** data by joining with reference tables
5. **Implement** data governance with PII masking and filtered views
6. **Handle** PII data with column masking and dynamic views
7. **Write** production-ready Silver Delta tables

---

### Silver Layer Pipeline

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border: 2px solid #1B3A4B; border-radius: 8px; padding: 20px; margin: 10px 0; font-family: monospace;">
<div style="text-align: center; font-weight: bold; font-size: 16px; margin-bottom: 15px; color: #1B3A4B;">SILVER LAYER TRANSFORMATION PIPELINE</div>
<div style="display: flex; align-items: center; justify-content: center; flex-wrap: wrap; gap: 8px;">
  <div style="background: #FFFFFF; border: 2px solid #137CBD; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #137CBD;">Bronze</div>
    <div style="font-size: 11px; color: #1B1F24;">Raw Data</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #FF3621; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #FF3621;">Profile</div>
    <div style="font-size: 11px; color: #1B1F24;">Understand</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #00A972; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #00A972;">Clean</div>
    <div style="font-size: 11px; color: #1B1F24;">Validate</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #137CBD; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #137CBD;">Enrich</div>
    <div style="font-size: 11px; color: #1B1F24;">Join & Calc</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #FFFFFF; border: 2px solid #FF3621; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #FF3621;">Protect PII</div>
    <div style="font-size: 11px; color: #1B1F24;">Mask & Govern</div>
  </div>
  <div style="font-size: 24px; color: #1B3A4B;">&rarr;</div>
  <div style="background: #1B3A4B; border: 2px solid #1B3A4B; border-radius: 8px; padding: 10px 14px; text-align: center; min-width: 100px;">
    <div style="font-weight: bold; color: #FFFFFF;">Silver</div>
    <div style="font-size: 11px; color: #F9F7F4;">Clean Data</div>
  </div>
</div>
</div>
""")

SILVER LAYER TRANSFORMATION PIPELINE 
 
 
 Bronze 
 Raw Data 
 
 → 
 
 Profile 
 Understand 
 
 → 
 
 Clean 
 Validate 
 
 → 
 
 Enrich 
 Join & Calc 
 
 → 
 
 Protect PII 
 Mask & Govern 
 
 → 
 
 Silver 
 Clean Data

---
## Step 0: Generate Data

> Executez cette cellule pour generer les donnees Bronze avec des problemes de qualite intentionnels.  
> Si vous avez deja fait le Lab 04, les donnees existent deja — cette cellule les recreera pour garantir l'idempotence.
>
> **Pas de Unity Catalog ni ADLS** — les donnees sont ecrites dans `/tmp/` et le Hive Metastore.

In [ ]:
# ============================================================
# STEP 0: Setup — connect to Bronze data from Lab 04
# No data generation needed — reads from Lab 04 Bronze table
# ============================================================

# Derive a unique database name per participant
_user = spark.sql("SELECT current_user()").first()[0]
_user_clean = _user.split("@")[0].replace(".", "_").replace("-", "_").lower()
USER_DB = f"training_lab_{_user_clean}"
print(f"Your personal database: {USER_DB}")

spark.sql(f"CREATE DATABASE IF NOT EXISTS {USER_DB}")

# Verify Bronze table from Lab 04 exists
try:
    _bronze_count = spark.table(f"{USER_DB}.bronze_placements").count()
    print(f"Bronze table found: {USER_DB}.bronze_placements ({_bronze_count} records)")
except Exception as e:
    print(f"ERROR: Bronze table not found! Run Lab 04 first.")
    print(f"Expected table: {USER_DB}.bronze_placements")
    raise Exception("Run Lab 04 (Bronze Layer) before starting Lab 05") from e

# Verify entities reference table
try:
    _ent_count = spark.table(f"{USER_DB}.axa_entities").count()
    print(f"Entities table found: {USER_DB}.axa_entities ({_ent_count} records)")
except Exception as e:
    print(f"ERROR: Entities table not found! Run Lab 04 first.")
    raise Exception("Run Lab 04 (Bronze Layer) before starting Lab 05") from e


In [ ]:
# Import required libraries
from pyspark.sql.functions import (
    col, count, sum as spark_sum, when, avg, min as spark_min, max as spark_max,
    to_date, year, month, round as spark_round,
    current_timestamp, lit, concat, sha2, regexp_replace
)
from pyspark.sql.types import *

# ---- Configuration (Hive Metastore, table-based) ----
DATABASE_NAME = USER_DB
BRONZE_TABLE = f"{DATABASE_NAME}.bronze_placements"
SILVER_TABLE = f"{DATABASE_NAME}.silver_placements"
ENTITIES_TABLE = f"{DATABASE_NAME}.axa_entities"

bronze_count = spark.table(BRONZE_TABLE).count()

print("=" * 60)
print("LAB 05 CONFIGURATION")
print("=" * 60)
print(f"Database:       {DATABASE_NAME}")
print(f"Bronze Table:   {BRONZE_TABLE}")
print(f"Silver Table:   {SILVER_TABLE}")
print(f"Entities Table: {ENTITIES_TABLE}")
print(f"Bronze records: {bronze_count}")

# Cleanup any leftover Silver table
try:
    spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
except:
    pass

print("Libraries imported successfully!")


In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">La couche Silver transforme les donnees brutes Bronze en donnees propres, enrichies et protegees. C'est ici que la qualite des donnees, les metriques business et la gouvernance sont appliquees.</span>
</div>
""")

A retenir : La couche Silver transforme les donnees brutes Bronze en donnees propres, enrichies et protegees. C'est ici que la qualite des donnees, les metriques business et la gouvernance sont appliquees.

---

## Section 2: Data Profiling

**POURQUOI :** Avant de nettoyer les donnees, il faut les comprendre. Le profilage revele les valeurs nulles, les aberrations, les formats invalides et les anomalies de distribution. Sans profilage, vous nettoyez a l'aveugle.

```
+-------------------------------------------------------------------+
|                    DATA PROFILING CHECKLIST                        |
+-------------------------------------------------------------------+
|                                                                   |
|   1. Schema check    - Are data types correct?                    |
|   2. Null analysis   - Which columns have missing values?         |
|   3. Range analysis  - Are numeric values within valid bounds?    |
|   4. Distinct values - Are categorical columns consistent?        |
|                                                                   |
+-------------------------------------------------------------------+
```

### Exercise 2.1: Read Bronze Data and Check Schema

Lisez la table Bronze creee dans le Lab 04 et inspectez sa structure.

In [ ]:
# Read Bronze data from Delta table
bronze_df = spark.table(BRONZE_TABLE)

# Display the schema
print("Bronze layer schema:")
bronze_df.printSchema()

print(f"\nTotal records in Bronze: {bronze_df.count()}")

# Preview the data
display(bronze_df.limit(5))

### Exercise 2.2: Null Value Analysis

Comptez les valeurs nulles dans chaque colonne critique.  
**TODO :** Remplissez les blancs avec `isNull()` pour completer les verifications de nulls.

In [ ]:
# Null value analysis across critical columns

null_counts = bronze_df.select(
    count("*").alias("total_records"),
    spark_sum(when(col("placement_id").isNull(), 1).otherwise(0)).alias("null_placement_id"),
    spark_sum(when(col("market_value").________(), 1).otherwise(0)).alias("null_market_value"),
    spark_sum(when(col("book_value").________(), 1).otherwise(0)).alias("null_book_value"),
    spark_sum(when(col("valuation_date").________(), 1).otherwise(0)).alias("null_valuation_date")
)

print("Null value analysis:")
display(null_counts)

In [ ]:
# Verification 2.2
assert null_counts.count() == 1, "Should produce exactly 1 summary row"
row = null_counts.first()
assert row.total_records > 0, "total_records should be > 0"
assert row.null_market_value > 0, "Expected some null market_value records from Step 0 data"
print(f"Exercise 2.2 passed! Null analysis complete - {row.total_records} records profiled")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">La methode qui verifie si une valeur est nulle : <code>isNull()</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Utilisez-la sur chaque colonne : <code>col(\"market_value\").isNull()</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Le pattern : <code>when(col(\"X\").isNull(), 1).otherwise(0)</code> compte les nulls</li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 La methode qui verifie si une valeur est nulle : isNull() 
 Utilisez-la sur chaque colonne : col(\"market_value\").isNull() 
 Le pattern : when(col(\"X\").isNull(), 1).otherwise(0) compte les nulls

### Exercise 2.3: Value Range Analysis

Examinez les min, max et moyennes pour les colonnes numeriques afin de detecter les aberrations.

In [ ]:
# Value range analysis for numeric columns
print("Value ranges for numeric columns:")
display(
    bronze_df.select(
        spark_min("market_value").alias("min_market_value"),
        spark_max("market_value").alias("max_market_value"),
        spark_round(avg("market_value"), 2).alias("avg_market_value"),
        spark_min("book_value").alias("min_book_value"),
        spark_max("book_value").alias("max_book_value"),
        spark_round(avg("book_value"), 2).alias("avg_book_value")
    )
)

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">Le profilage des donnees est la premiere etape de tout pipeline Silver. Il vous indique <em>quoi</em> nettoyer, <em>ou</em> sont les problemes, et <em>combien</em> de donnees vous allez perdre. Ne sautez jamais cette etape.</span>
</div>
""")

A retenir : Le profilage des donnees est la premiere etape de tout pipeline Silver. Il vous indique quoi nettoyer, ou sont les problemes, et combien de donnees vous allez perdre. Ne sautez jamais cette etape.

---

## Section 3: Data Cleaning

Maintenant que nous comprenons les problemes de qualite, appliquons les regles de nettoyage :

```
+-------------------------------------------------------------------+
|                    DATA QUALITY RULES                              |
+-------------------------------------------------------------------+
|                                                                   |
|   RULE 1: NOT NULL CHECKS                                         |
|   - placement_id    must not be null (primary key)                |
|   - market_value    must not be null (required for calculations)  |
|   - book_value      must not be null (required for calculations)  |
|   - valuation_date  must not be null (required for partitioning)  |
|                                                                   |
|   RULE 2: RANGE VALIDATION                                        |
|   - market_value   >= 0  (no negative valuations)                 |
|   - book_value     > 0   (avoid division by zero)                 |
|                                                                   |
+-------------------------------------------------------------------+
```

### Exercise 3.1: Filter Null Values

Supprimez les enregistrements avec des valeurs nulles dans les colonnes critiques.  
**TODO :** Remplissez les blancs avec la methode opposee a `isNull()` pour conserver les enregistrements valides.

In [ ]:
# Filter out records with null values in critical columns

clean_df = bronze_df.filter(
    (col("placement_id").isNotNull()) &
    (col("market_value").________()) &
    (col("book_value").________()) &
    (col("valuation_date").________())
)

original_count = bronze_df.count()
after_null_filter = clean_df.count()

print(f"Original records:       {original_count}")
print(f"After null filtering:   {after_null_filter}")
print(f"Records removed (nulls): {original_count - after_null_filter}")

In [ ]:
# Verification 3.1
assert clean_df.count() > 0, "All records filtered out - check isNotNull() syntax"
assert clean_df.count() < original_count, "No records removed - nulls should have been filtered"
null_check = clean_df.filter(col("market_value").isNull()).count()
assert null_check == 0, f"Still {null_check} null market_value records - check isNotNull()"
print(f"Exercise 3.1 passed! {clean_df.count()} records after null filtering (removed {original_count - clean_df.count()})")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">L'oppose de <code>isNull()</code> est <code>isNotNull()</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Combinez les conditions avec <code>&amp;</code> (AND logique)</li>
      <li style="margin: 6px 0; color: #1B1F24;">Exemple : <code>col(\"market_value\").isNotNull()</code></li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 L'oppose de isNull() est isNotNull() 
 Combinez les conditions avec & (AND logique) 
 Exemple : col(\"market_value\").isNotNull()

### Exercise 3.2: Range Validation

Assurez-vous que les valeurs numeriques sont dans des plages acceptables.  
**TODO :** Remplissez les blancs avec les valeurs seuils correctes pour la validation de plage.

In [ ]:
# Apply range validation rules

validated_df = clean_df.filter(
    (col("market_value") >= ________)  # no negative valuations &
    (col("book_value") > ________)  # avoid division by zero
)

after_range_filter = validated_df.count()

# Data quality summary
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)
print(f"Original Bronze records:      {original_count}")
print(f"After null filtering:         {after_null_filter}")
print(f"After range validation:       {after_range_filter}")
print(f"Total records removed:        {original_count - after_range_filter}")
print(f"Quality rate:                 {(after_range_filter / original_count) * 100:.2f}%")

In [ ]:
# Verification 3.2
assert validated_df.count() > 0, "All records filtered - check range conditions"
assert validated_df.count() <= after_null_filter, "Range filter should not add records"
neg_check = validated_df.filter(col("market_value") < 0).count()
assert neg_check == 0, f"Found {neg_check} negative market_value records - check >= 0 filter"
print(f"Exercise 3.2 passed! {validated_df.count()} records after range validation")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">La valeur de marche ne peut pas etre negative : <code>&gt;= 0</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">La valeur comptable doit etre strictement positive (pour eviter la division par zero) : <code>&gt; 0</code></li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 La valeur de marche ne peut pas etre negative : >= 0 
 La valeur comptable doit etre strictement positive (pour eviter la division par zero) : > 0

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">Le nettoyage des donnees s'appuie sur des <em>regles metier</em>, pas des filtres arbitraires. Chaque regle doit etre documentee et justifiee. Suivez le nombre d'enregistrements supprimes a chaque etape pour detecter les problemes en amont.</span>
</div>
""")

A retenir : Le nettoyage des donnees s'appuie sur des regles metier , pas des filtres arbitraires. Chaque regle doit etre documentee et justifiee. Suivez le nombre d'enregistrements supprimes a chaque etape pour detecter les problemes en amont.

---

## Section 4: Type Casting and Date Handling

Les donnees brutes arrivent souvent avec des types incorrects (ex: dates en string). Il faut les convertir pour les analyses et le partitionnement.

```
+-------------------------------------------------------------------+
|   BEFORE                              AFTER                       |
|   valuation_date   STRING   --->    valuation_date   DATE         |
|                                                                   |
|   EXTRACTED COMPONENTS:                                           |
|   valuation_year    INT     <---    year(valuation_date)          |
|   valuation_month   INT     <---    month(valuation_date)         |
+-------------------------------------------------------------------+
```

### Exercise 4.1: Cast Valuation Date

Convertissez `valuation_date` de string en type date.  
**TODO :** Remplissez le blanc avec la fonction qui convertit une string en date.

In [ ]:
# Cast valuation_date from string to date type

typed_df = validated_df.withColumn(
    "valuation_date",
    ________(col("valuation_date"))  # convert string to date
)

# Verify the type change
print("Schema after type casting:")
typed_df.select("valuation_date").printSchema()

In [ ]:
# Verification 4.1
from pyspark.sql.types import DateType as _DateType
vd_type = typed_df.schema["valuation_date"].dataType
assert isinstance(vd_type, _DateType), f"valuation_date should be DateType, got {vd_type} - use to_date()"
print(f"Exercise 4.1 passed! valuation_date is now DateType")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">La fonction qui convertit une string en date : <code>to_date()</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Usage : <code>to_date(col(\"valuation_date\"))</code></li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 La fonction qui convertit une string en date : to_date() 
 Usage : to_date(col(\"valuation_date\"))

### Exercise 4.2: Extract Year and Month Components

Extrayez l'annee et le mois pour l'analyse temporelle et le partitionnement.  
**TODO :** Remplissez les blancs avec les fonctions qui extraient l'annee et le mois.

In [ ]:
# Extract valuation_year and valuation_month from valuation_date

dated_df = typed_df \
    .withColumn("valuation_year", ________(col("valuation_date"))  # extract year) \
    .withColumn("valuation_month", ________(col("valuation_date"))  # extract month)

# Display sample results
print("Date components extracted:")
display(
    dated_df.select(
        "placement_id",
        "valuation_date",
        "valuation_year",
        "valuation_month"
    ).limit(5)
)

In [ ]:
# Verification 4.2
assert "valuation_year" in dated_df.columns, "valuation_year missing - use year()"
assert "valuation_month" in dated_df.columns, "valuation_month missing - use month()"
sample = dated_df.select("valuation_year", "valuation_month").first()
assert 2020 <= sample.valuation_year <= 2030, f"Year {sample.valuation_year} looks wrong"
assert 1 <= sample.valuation_month <= 12, f"Month {sample.valuation_month} looks wrong"
print(f"Exercise 4.2 passed! Sample: year={sample.valuation_year}, month={sample.valuation_month}")

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">La conversion de types assure la coherence dans tout le pipeline. Extraire les composantes de date (annee, mois) est essentiel pour le partitionnement, l'analyse temporelle et des requetes performantes.</span>
</div>
""")

A retenir : La conversion de types assure la coherence dans tout le pipeline. Extraire les composantes de date (annee, mois) est essentiel pour le partitionnement, l'analyse temporelle et des requetes performantes.

---

## Section 5: Business Metrics (Calculated Columns)

La couche Silver est l'endroit ou les donnees brutes deviennent *utiles*. Nous derivons des metriques business que les analystes et les tables Gold utilisent.

**Pourquoi ces metriques pour AXA ?**
- Le **gain/perte non realise** mesure la performance du portefeuille d'investissement
- Le **pourcentage de gain/perte** normalise le rendement pour comparer des investissements de tailles differentes
- Ces metriques sont essentielles pour le reporting reglementaire (Solvency II) et la gestion des risques

```
+-------------------------------------------------------------------+
|                    CALCULATED BUSINESS METRICS                     |
+-------------------------------------------------------------------+
|                                                                   |
|   METRIC 1: UNREALIZED GAIN/LOSS                                  |
|   unrealized_gain_loss = market_value - book_value                |
|   * Positive = Unrealized GAIN                                    |
|   * Negative = Unrealized LOSS                                    |
|                                                                   |
|   METRIC 2: GAIN/LOSS PERCENTAGE                                  |
|   gain_loss_pct = (market_value - book_value) / book_value * 100  |
|   * Measures return as percentage of original investment          |
|                                                                   |
+-------------------------------------------------------------------+
```

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Investment Portfolio Metrics</h3>
  <div style="display: flex; justify-content: center; gap: 20px; flex-wrap: wrap; margin-top: 15px;">
    <div style="background: #FFFFFF; border: 2px solid #00A972; border-radius: 8px; padding: 15px 20px; min-width: 200px; text-align: center;">
      <div style="font-size: 13px; color: #757575;">Unrealized Gain/Loss</div>
      <div style="font-size: 16px; font-weight: bold; color: #00A972; margin: 8px 0;">market_value - book_value</div>
      <div style="font-size: 12px; color: #1B1F24;">Positive = Gain | Negative = Loss</div>
    </div>
    <div style="background: #FFFFFF; border: 2px solid #137CBD; border-radius: 8px; padding: 15px 20px; min-width: 200px; text-align: center;">
      <div style="font-size: 13px; color: #757575;">Gain/Loss %</div>
      <div style="font-size: 16px; font-weight: bold; color: #137CBD; margin: 8px 0;">(MV - BV) / BV &times; 100</div>
      <div style="font-size: 12px; color: #1B1F24;">Normalized return on investment</div>
    </div>
  </div>
  <div style="background: #FFFFFF; border-left: 4px solid #FF3621; border-radius: 4px; padding: 12px 16px; margin-top: 15px;">
    <strong style="color: #FF3621;">Business Context:</strong> <span style="color: #1B1F24;">Pour une compagnie d'assurance comme AXA, ces metriques servent au suivi de la performance des placements, au reporting Solvency II, et a la gestion des risques de marche.</span>
  </div>
</div>
""")

Investment Portfolio Metrics 
 
 
 Unrealized Gain/Loss 
 market_value - book_value 
 Positive = Gain | Negative = Loss 
 
 
 Gain/Loss % 
 (MV - BV) / BV × 100 
 Normalized return on investment 
 
 
 
 Business Context: Pour une compagnie d'assurance comme AXA, ces metriques servent au suivi de la performance des placements, au reporting Solvency II, et a la gestion des risques de marche.

### Exercise 5.1: Calculate Unrealized Gain/Loss

**TODO :** Remplissez le blanc avec le nom de la colonne a soustraire.

In [ ]:
# Calculate unrealized gain/loss = market_value - book_value

metrics_df = dated_df.withColumn(
    "unrealized_gain_loss",
    col("market_value") - col("________")  # what did we pay for it?
)

# Preview the result
print("Unrealized Gain/Loss calculated:")
display(
    metrics_df.select(
        "placement_id",
        "market_value",
        "book_value",
        "unrealized_gain_loss"
    ).limit(5)
)

In [ ]:
# Verification 5.1
assert "unrealized_gain_loss" in metrics_df.columns, "unrealized_gain_loss column missing"
sample = metrics_df.select("market_value", "book_value", "unrealized_gain_loss").first()
expected = sample.market_value - sample.book_value
assert abs(sample.unrealized_gain_loss - expected) < 0.01, "Formula incorrect - should be market_value - book_value"
print(f"Exercise 5.1 passed! Sample: {sample.market_value:.2f} - {sample.book_value:.2f} = {sample.unrealized_gain_loss:.2f}")

### Exercise 5.2: Calculate Gain/Loss Percentage

Calculez le gain/perte en pourcentage de la valeur comptable, arrondi a 2 decimales.  
**TODO :** Remplissez les blancs avec le diviseur et le multiplicateur.

In [ ]:
# Calculate gain/loss percentage

metrics_df = metrics_df.withColumn(
    "gain_loss_pct",
    spark_round(
        (col("market_value") - col("book_value")) / col("________") * ________  # divide by original, multiply for percentage,
        2
    )
)

# Preview the result
print("Gain/Loss Percentage calculated:")
display(
    metrics_df.select(
        "placement_id",
        "market_value",
        "book_value",
        "unrealized_gain_loss",
        "gain_loss_pct"
    ).limit(5)
)

In [ ]:
# Verification 5.2
assert "gain_loss_pct" in metrics_df.columns, "gain_loss_pct column missing"
sample = metrics_df.select("market_value", "book_value", "gain_loss_pct").first()
expected_pct = round((sample.market_value - sample.book_value) / sample.book_value * 100, 2)
assert abs(sample.gain_loss_pct - expected_pct) < 0.1, "Percentage formula incorrect"
print(f"Exercise 5.2 passed! Sample gain/loss: {sample.gain_loss_pct}%")

In [ ]:
# Analyze gain/loss distribution: gainers vs losers vs neutral
print("Gain/Loss Distribution:")
display(
    metrics_df.select(
        count("*").alias("total_placements"),
        spark_sum(when(col("unrealized_gain_loss") > 0, 1).otherwise(0)).alias("gainers"),
        spark_sum(when(col("unrealized_gain_loss") < 0, 1).otherwise(0)).alias("losers"),
        spark_sum(when(col("unrealized_gain_loss") == 0, 1).otherwise(0)).alias("neutral"),
        spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct"),
        spark_round(spark_min("gain_loss_pct"), 2).alias("min_gain_loss_pct"),
        spark_round(spark_max("gain_loss_pct"), 2).alias("max_gain_loss_pct")
    )
)

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">Les colonnes calculees transforment les donnees brutes en metriques business. La couche Silver est ou cette creation de valeur a lieu. Validez toujours vos formules (ex: division par zero) avant de les appliquer a l'echelle.</span>
</div>
""")

A retenir : Les colonnes calculees transforment les donnees brutes en metriques business. La couche Silver est ou cette creation de valeur a lieu. Validez toujours vos formules (ex: division par zero) avant de les appliquer a l'echelle.

---

## Section 6: Data Enrichment (Join with Reference Data)

L'enrichissement ajoute du contexte a partir de tables de reference (dimensions). Ici, nous joignons les placements avec la table de reference des entites pour ajouter les noms et les regions.

```
+-------------------------------------------------------------------+
|                     DATA ENRICHMENT JOIN                          |
+-------------------------------------------------------------------+
|                                                                   |
|   PLACEMENTS TABLE              ENTITIES REFERENCE                |
|   +-------------------+         +-------------------+             |
|   | placement_id     |         | entity_id        |             |
|   | axa_entity_id ---|----+    | entity_name      |             |
|   | asset_class      |    |    | region           |             |
|   | market_value     |    |    | country          |             |
|   | book_value       |    |    +-------------------+             |
|   | ...              |    |             ^                        |
|   +-------------------+    +------------|                        |
|                                         |                        |
|                            JOIN ON: axa_entity_id = entity_id    |
|                                                                   |
+-------------------------------------------------------------------+
```

In [ ]:
# Load entities reference data from Hive table
entities_df = spark.table(f"{DATABASE_NAME}.axa_entities")

print(f"Entities reference records: {entities_df.count()}")
display(entities_df)

### Exercise 6.1: Join Placements with Entities

Enrichissez les donnees de placements en joignant avec la table de reference des entites.  
**TODO :** Remplissez les blancs avec la colonne de jointure et le type de jointure.

In [ ]:
# Join placements with entities reference

enriched_df = metrics_df.join(
    entities_df,
    metrics_df.axa_entity_id == entities_df.________  # join column on entities side,
    "________"  # keep all placements even if no match
)

# Display enriched data
print("Enriched placements data:")
display(
    enriched_df.select(
        "placement_id",
        "axa_entity_id",
        "entity_name",
        "region",
        "asset_class",
        "market_value"
    ).limit(10)
)

In [ ]:
# Verification 6.1
assert "entity_name" in enriched_df.columns, "entity_name missing - check join column (entity_id)"
assert "region" in enriched_df.columns, "region missing - check join type"
assert enriched_df.count() > 0, "Join produced no results"
matched = enriched_df.filter(col("entity_name").isNotNull()).count()
print(f"Exercise 6.1 passed! {enriched_df.count()} enriched records ({matched} matched entities)")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">La colonne de jointure cote entites : <code>entity_id</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Utilisez un <code>\"left\"</code> join pour conserver tous les placements meme sans correspondance</li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 La colonne de jointure cote entites : entity_id 
 Utilisez un \"left\" join pour conserver tous les placements meme sans correspondance

In [ ]:
# Check for unmatched entities (placements with no matching reference)
unmatched = enriched_df.filter(col("entity_name").isNull())
unmatched_count = unmatched.count()

print(f"Unmatched entity records: {unmatched_count}")

if unmatched_count > 0:
    print("\nUnmatched entity IDs:")
    display(
        unmatched.select("axa_entity_id")
        .distinct()
        .limit(10)
    )

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">L'enrichissement par jointure ajoute du contexte metier aux donnees brutes. Utilisez toujours un <code>left</code> join pour conserver tous les enregistrements, puis verifiez les lignes sans correspondance. Les enregistrements non-apparies indiquent des lacunes dans les donnees de reference a signaler en amont.</span>
</div>
""")

A retenir : L'enrichissement par jointure ajoute du contexte metier aux donnees brutes. Utilisez toujours un left join pour conserver tous les enregistrements, puis verifiez les lignes sans correspondance. Les enregistrements non-apparies indiquent des lacunes dans les donnees de reference a signaler en amont.

---

## Section 7: Gouvernance des Donnees & Protection des PII

En production, tout le monde ne devrait pas voir toutes les donnees. Les **PII** (Personally Identifiable Information) comme les noms, emails et identifiants doivent etre proteges.

Dans cette section, nous demonstrons des techniques de gouvernance qui fonctionnent dans le **Hive Metastore** :
- **Masquage PII** : SHA2 et REGEXP_REPLACE dans des vues SQL (fonctionne en Hive)
- **Securite au niveau des lignes** : Vues filtrees par region (approche Hive)
- **GRANT** : Controle d'acces de base (montre en exemples)

> **Note Unity Catalog :** Avec Unity Catalog, vous beneficiez de fonctionnalites supplementaires : filtres de lignes natifs (`ALTER TABLE SET ROW FILTER`), fonctions de masquage de colonnes, et gouvernance centralisee sur plusieurs catalogues. Les techniques montrees ici sont les equivalents Hive Metastore.

In [1]:
displayHTML("""
<div style="background-color: #F9F7F4; border: 2px solid #1B3A4B; border-radius: 8px; padding: 20px; margin: 10px 0; font-family: monospace;">
<div style="text-align: center; font-weight: bold; font-size: 16px; margin-bottom: 15px; color: #1B3A4B;">DATA GOVERNANCE MODEL</div>

<div style="margin-bottom: 15px;">
<strong style="color: #1B1F24;">2-Level Namespace (Hive Metastore):</strong>
<div style="display: flex; align-items: center; justify-content: center; gap: 8px; margin-top: 8px;">
  <div style="background: #FFFFFF; border: 2px solid #00A972; border-radius: 6px; padding: 8px 16px; text-align: center;">
    <div style="font-weight: bold; color: #00A972;">Database</div>
    <div style="font-size: 11px; color: #1B1F24;">e.g. training_lab</div>
  </div>
  <div style="font-size: 20px; color: #1B3A4B;">.</div>
  <div style="background: #FFFFFF; border: 2px solid #FF3621; border-radius: 6px; padding: 8px 16px; text-align: center;">
    <div style="font-weight: bold; color: #FF3621;">Table</div>
    <div style="font-size: 11px; color: #1B1F24;">e.g. silver_placements</div>
  </div>
</div>
</div>

<div style="margin-bottom: 10px;">
<strong style="color: #1B1F24;">Row-Level Security (via filtered views):</strong>
<div style="background: #FFFFFF; border: 1px solid #1B3A4B; border-radius: 4px; padding: 8px; margin-top: 6px; font-size: 12px;">
  <div style="color: #1B1F24;"><span style="color: #137CBD; font-weight: bold;">europe_team</span> &rarr; VIEW with WHERE region = 'Europe'</div>
  <div style="color: #1B1F24;"><span style="color: #00A972; font-weight: bold;">americas_team</span> &rarr; VIEW with WHERE region = 'Americas'</div>
  <div style="color: #1B1F24;"><span style="color: #FF3621; font-weight: bold;">global_admin</span> &rarr; direct table access (all rows)</div>
</div>
</div>

<div style="margin-bottom: 10px;">
<strong style="color: #1B1F24;">Column Masking (via masked views):</strong>
<div style="background: #FFFFFF; border: 1px solid #1B3A4B; border-radius: 4px; padding: 8px; margin-top: 6px; font-size: 12px;">
  <div style="color: #1B1F24;"><span style="color: #FF3621; font-weight: bold;">trader_name</span> &rarr; SHA2 hash: <code>a3f2c8...</code></div>
  <div style="color: #1B1F24;"><span style="color: #FF3621; font-weight: bold;">trader_email</span> &rarr; regex mask: <code>****@company.com</code></div>
</div>
</div>

<div style="background: #E8F5E9; border-radius: 4px; padding: 8px; font-size: 11px; color: #1B1F24;">
  <strong>Unity Catalog upgrade:</strong> Ajoute des filtres de lignes natifs, le masquage de colonnes integre, et la gouvernance centralisee multi-catalogues.
</div>
</div>
""")

DATA GOVERNANCE MODEL 

 
 2-Level Namespace (Hive Metastore): 
 
 
 Database 
 e.g. training_lab 
 
 . 
 
 Table 
 e.g. silver_placements 
 
 
 

 
 Row-Level Security (via filtered views): 
 
 europe_team → VIEW with WHERE region = 'Europe' 
 americas_team → VIEW with WHERE region = 'Americas' 
 global_admin → direct table access (all rows) 
 
 

 
 Column Masking (via masked views): 
 
 trader_name → SHA2 hash: a3f2c8... 
 trader_email → regex mask: ****@company.com 
 
 

 
 Unity Catalog upgrade: Ajoute des filtres de lignes natifs, le masquage de colonnes integre, et la gouvernance centralisee multi-catalogues.

### Exercise 7.1: Write Silver Table

Selectionnez les colonnes finales, ajoutez un timestamp de traitement, et ecrivez la table Silver Delta.

In [ ]:
# Select final columns for Silver table
silver_df = enriched_df.select(
    col("placement_id"),
    col("axa_entity_id"),
    col("entity_name"),
    col("region"),
    col("country"),
    col("asset_class"),
    col("instrument_id"),
    col("market_value"),
    col("book_value"),
    col("unrealized_gain_loss"),
    col("gain_loss_pct"),
    col("currency"),
    col("valuation_date"),
    col("valuation_year"),
    col("valuation_month"),
    current_timestamp().alias("_processing_timestamp")
)

# Write to Silver Delta table
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("valuation_year") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

# Register as Hive table

silver_count = spark.table(f"{DATABASE_NAME}.silver_placements").count()
print(f"Silver table written to: {SILVER_TABLE}")
print(f"Table registered: {DATABASE_NAME}.silver_placements")
print(f"Records: {silver_count}")

In [ ]:
# Verification 7.1
silver_count = spark.table(f"{DATABASE_NAME}.silver_placements").count()
assert silver_count > 0, "Silver table is empty"
silver_cols = spark.table(f"{DATABASE_NAME}.silver_placements").columns
assert "unrealized_gain_loss" in silver_cols, "Missing business metric columns"
assert "entity_name" in silver_cols, "Missing enrichment columns"
assert "valuation_year" in silver_cols, "Missing date component columns"
print(f"Exercise 7.1 passed! Silver table has {silver_count} records with {len(silver_cols)} columns")

### Exercise 7.2: PII Masking with SHA2 Hash

Nous simulons des colonnes PII et creons des **vues masquees** qui protegent les donnees sensibles.  
SHA2 et REGEXP_REPLACE fonctionnent dans les vues du Hive Metastore.  
**TODO :** Remplissez les blancs avec les fonctions SQL de masquage.

In [ ]:
# Simulate adding PII columns for demonstration
df_with_pii = spark.table(f"{DATABASE_NAME}.silver_placements") \
    .withColumn("trader_name", concat(lit("Trader_"), col("placement_id"))) \
    .withColumn("trader_email", concat(col("placement_id"), lit("@axa.com")))

print("Sample data WITH PII (visible to authorized users only):")
display(
    df_with_pii.select(
        "placement_id", "trader_name", "trader_email", "market_value"
    ).limit(5)
)

In [ ]:
# Create a masked view - SHA2 and REGEXP_REPLACE work in Hive Metastore

spark.sql(f"""
    CREATE OR REPLACE VIEW {DATABASE_NAME}.placements_masked AS
    SELECT 
        placement_id,
        axa_entity_id,
        entity_name,
        asset_class,
        market_value,
        book_value,
        -- PII columns masked using SHA2 hash and regex
        ________(CONCAT('Trader_', placement_id), 256) AS trader_name_masked,
        ________(CONCAT(placement_id, '@axa.com'), '(^[^@]+)', '****') AS trader_email_masked,
        valuation_date
    FROM {DATABASE_NAME}.silver_placements
""")

print("Masked view created successfully.")
print(f"\nSample from {DATABASE_NAME}.placements_masked:")
display(spark.sql(f"SELECT * FROM {DATABASE_NAME}.placements_masked LIMIT 5"))

In [ ]:
# Verification 7.2
masked_df = spark.table(f"{DATABASE_NAME}.placements_masked")
assert "trader_name_masked" in masked_df.columns, "Masked view missing trader_name_masked"
assert "trader_email_masked" in masked_df.columns, "Masked view missing trader_email_masked"
sample = masked_df.first()
assert "****" in sample.trader_email_masked, "Email not masked - check REGEXP_REPLACE"
assert len(sample.trader_name_masked) == 64, f"SHA2(256) should produce 64-char hex string, got {len(sample.trader_name_masked)}"
print(f"Exercise 7.2 passed! PII masking verified (SHA2 hash length: {len(sample.trader_name_masked)})")

In [1]:
displayHTML("""
<details style="margin: 10px 0;">
  <summary style="cursor: pointer; color: #1B3A4B; font-weight: bold; padding: 8px 0;">Cliquez pour reveler les indices</summary>
  <div style="background: #F9F7F4; padding: 12px 16px; border-radius: 8px; margin-top: 8px; border-left: 4px solid #F9A825;">
    <ul style="margin: 8px 0; padding-left: 20px;">
      <li style="margin: 6px 0; color: #1B1F24;">Pour un hash irreversible : <code>SHA2(..., 256)</code></li>
      <li style="margin: 6px 0; color: #1B1F24;">Pour masquer partiellement un email : <code>REGEXP_REPLACE(..., '(^[^@]+)', '****')</code></li>
    </ul>
  </div>
</details>
""")

Cliquez pour reveler les indices 
 
 
 Pour un hash irreversible : SHA2(..., 256) 
 Pour masquer partiellement un email : REGEXP_REPLACE(..., '(^[^@]+)', '****')

### Exercise 7.3: Row-Level Security via Filtered Views

Dans le Hive Metastore, nous implementons la securite au niveau des lignes en creant des vues filtrees. Chaque vue restreint les donnees a une region specifique.

> **Equivalent UC :** Avec Unity Catalog, vous utiliseriez `ALTER TABLE SET ROW FILTER` pour un filtrage automatique et transparent. L'approche par vues montree ici atteint le meme resultat mais necessite une gestion explicite des vues.

In [ ]:
# Create region-specific filtered views
spark.sql(f"""
    CREATE OR REPLACE VIEW {DATABASE_NAME}.placements_europe AS
    SELECT * FROM {DATABASE_NAME}.silver_placements
    WHERE region = 'Europe'
""")

spark.sql(f"""
    CREATE OR REPLACE VIEW {DATABASE_NAME}.placements_americas AS
    SELECT * FROM {DATABASE_NAME}.silver_placements
    WHERE region = 'Americas'
""")

europe_count = spark.table(f"{DATABASE_NAME}.placements_europe").count()
americas_count = spark.table(f"{DATABASE_NAME}.placements_americas").count()
total_count = spark.table(f"{DATABASE_NAME}.silver_placements").count()

print(f"Europe-only view:   {europe_count} records")
print(f"Americas-only view: {americas_count} records")
print(f"Total Silver:       {total_count} records")
print(f"\nEn production, chaque equipe ne voit que sa vue regionale.")

### Exercise 7.4: Access Control (GRANT Demonstration)

Dans le Hive Metastore, vous pouvez utiliser des instructions `GRANT` basiques sur les tables et vues.  
Les exemples suivants montrent a quoi ressemble le controle d'acces en production.

> **Upgrade UC :** Unity Catalog ajoute la gouvernance centralisee, les grants au niveau colonne, et l'audit logging. Avec le Hive Metastore, les grants sont limites au workspace.

In [ ]:
# Access control demonstration
# Ces GRANT fonctionnent si Table Access Control est active sur le workspace

grant_statements = """
-- Donner aux analystes l'acces en lecture a la table Silver :
-- GRANT SELECT ON TABLE training_lab.silver_placements TO `data_analysts_group`;

-- Donner aux auditeurs l'acces a la vue masquee uniquement :
-- GRANT SELECT ON TABLE training_lab.placements_masked TO `external_auditors`;

-- Donner a l'equipe Europe l'acces a leur vue regionale :
-- GRANT SELECT ON TABLE training_lab.placements_europe TO `europe_team`;

-- Voir les permissions actuelles :
-- SHOW GRANT ON TABLE training_lab.silver_placements;
"""

print("HIVE METASTORE GRANT STATEMENTS (for production use):")
print(grant_statements)

print("NOTE: Ces instructions necessitent que le Table Access Control soit active.")
print("Avec Unity Catalog, la gouvernance s'etend a travers les catalogues avec des permissions fines.")

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">La gouvernance des donnees consiste a donner aux <em>bonnes personnes</em> acces aux <em>bonnes donnees</em>. Dans le Hive Metastore, utilisez SHA2/REGEXP_REPLACE dans les vues pour le masquage PII, les vues filtrees pour la securite au niveau des lignes, et les GRANT pour le controle d'acces. Unity Catalog ajoute des filtres natifs, le masquage integre et la gouvernance centralisee.</span>
</div>
""")

A retenir : La gouvernance des donnees consiste a donner aux bonnes personnes acces aux bonnes donnees . Dans le Hive Metastore, utilisez SHA2/REGEXP_REPLACE dans les vues pour le masquage PII, les vues filtrees pour la securite au niveau des lignes, et les GRANT pour le controle d'acces. Unity Catalog ajoute des filtres natifs, le masquage integre et la gouvernance centralisee.

---

## Section 8: Verification

Validez la couche Silver et examinez les resultats des transformations.

In [ ]:
# Check Silver table record count and preview
silver_result = spark.table(f"{DATABASE_NAME}.silver_placements")
print(f"Total records in Silver table: {silver_result.count()}")
print("\nSilver table schema:")
silver_result.printSchema()
display(silver_result.limit(5))

In [ ]:
# Analyze by asset_class
print("Performance by Asset Class:")
display(
    spark.table(f"{DATABASE_NAME}.silver_placements")
    .groupBy("asset_class")
    .agg(
        count("*").alias("num_placements"),
        spark_round(spark_sum("market_value"), 2).alias("total_market_value"),
        spark_round(spark_sum("unrealized_gain_loss"), 2).alias("total_gain_loss"),
        spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct")
    )
    .orderBy("total_market_value", ascending=False)
)

In [ ]:
# Analyze by entity_name
print("Performance by Entity:")
display(
    spark.table(f"{DATABASE_NAME}.silver_placements")
    .groupBy("entity_name", "region")
    .agg(
        count("*").alias("num_placements"),
        spark_round(spark_sum("market_value"), 2).alias("total_market_value"),
        spark_round(avg("gain_loss_pct"), 2).alias("avg_gain_loss_pct")
    )
    .orderBy("total_market_value", ascending=False)
    .limit(10)
)

In [ ]:
# View Delta table history
print("Silver table history:")
display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}"))

In [1]:
displayHTML("""
<div style="background: #FFFFFF; border-left: 5px solid #00A972; border-radius: 4px; padding: 15px; margin: 15px 0;">
  <strong style="color: #00A972;">A retenir :</strong> <span style="color: #1B1F24;">Verifiez toujours votre couche Silver avec des requetes d'aggregation et l'historique Delta. Cela confirme que le nettoyage, l'enrichissement et la gouvernance ont ete correctement appliques avant que les tables Gold ne consomment les donnees.</span>
</div>
""")

A retenir : Verifiez toujours votre couche Silver avec des requetes d'aggregation et l'historique Delta. Cela confirme que le nettoyage, l'enrichissement et la gouvernance ont ete correctement appliques avant que les tables Gold ne consomment les donnees.

---

## Ce que vous avez appris

In [1]:
displayHTML("""
<div style="background: #F9F7F4; border-radius: 12px; padding: 25px; margin: 20px 0; font-family: 'Segoe UI', sans-serif;">
  <h3 style="text-align: center; margin-top: 0; color: #1B3A4B;">Lab 05 Summary</h3>
  <table style="width: 100%; border-collapse: collapse; background: #FFFFFF; border-radius: 8px; overflow: hidden;">
    <thead>
      <tr style="background: #1B3A4B; color: #FFFFFF;">
        <th style="padding: 12px 15px; text-align: left;">Concept</th>
        <th style="padding: 12px 15px; text-align: left;">Ce que vous avez appris</th>
      </tr>
    </thead>
    <tbody>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">Data Profiling</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Toujours comprendre les donnees avant de nettoyer — profiler les nulls, plages et distributions</td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">Null Handling</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Filtrer ou remplir les nulls selon les regles metier ; suivre le nombre de suppressions</td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">Type Casting</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Assurer la coherence des types (string vers date, extraction annee/mois)</td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">Business Metrics</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Deriver de la valeur avec des colonnes calculees (gain/perte, pourcentages)</td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">PII Masking</td>
        <td style="padding: 10px 15px; color: #1B1F24;">SHA2 pour masquage irreversible ; REGEXP_REPLACE pour masquage partiel des emails</td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0; background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">Row-Level Security</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Restreindre l'acces par region via des vues filtrees (UC : row filters natifs)</td>
      </tr>
      <tr style="border-bottom: 1px solid #E0E0E0;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">Dynamic Views</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Creer des vues masquees/filtrees pour differents niveaux sans dupliquer les donnees</td>
      </tr>
      <tr style="background: #F9F7F4;">
        <td style="padding: 10px 15px; font-weight: bold; color: #9E9E9E;">Orchestration</td>
        <td style="padding: 10px 15px; color: #1B1F24;">Chainer Bronze et Silver avec des dependances via Databricks Workflows</td>
      </tr>
    </tbody>
  </table>
</div>
""")

Concept,Ce que vous avez appris
Data Profiling,"Toujours comprendre les donnees avant de nettoyer — profiler les nulls, plages et distributions"
Null Handling,Filtrer ou remplir les nulls selon les regles metier ; suivre le nombre de suppressions
Type Casting,"Assurer la coherence des types (string vers date, extraction annee/mois)"
Business Metrics,"Deriver de la valeur avec des colonnes calculees (gain/perte, pourcentages)"
PII Masking,SHA2 pour masquage irreversible ; REGEXP_REPLACE pour masquage partiel des emails
Row-Level Security,Restreindre l'acces par region via des vues filtrees (UC : row filters natifs)
Dynamic Views,Creer des vues masquees/filtrees pour differents niveaux sans dupliquer les donnees
Orchestration,Chainer Bronze et Silver avec des dependances via Databricks Workflows


### Next Steps

Passez au **Lab 06 - Gold Layer** ou vous allez :
- Lire les donnees Silver que vous venez de creer
- Construire des **aggregations business** (par classe d'actif, par entite, par region)
- Creer des **tables Gold** pretes pour l'analyse et le reporting
- Implementer des vues analytiques pour les tableaux de bord

---
## Cleanup

In [ ]:
# Cleanup (optional -- uncomment to reset)
# spark.sql(f"DROP VIEW IF EXISTS {DATABASE_NAME}.placements_masked")
# spark.sql(f"DROP VIEW IF EXISTS {DATABASE_NAME}.placements_europe")
# spark.sql(f"DROP VIEW IF EXISTS {DATABASE_NAME}.placements_americas")
# spark.sql(f"DROP TABLE IF EXISTS {DATABASE_NAME}.silver_placements")
# dbutils.fs.rm(f"{USER_DB}/silver", recurse=True)
# print("Lab 05 cleanup complete.")